In [17]:
# -------------------------
# Modified Comparison Code
# -------------------------
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import torch.nn.functional as F  # Add this line at the top
import os



In [ ]:
# Configuration (Keep Original)
MOBILITY = 120
COMPRESSION_RATIO = '1/8'
INPUT_DIM = 256
OUTPUT_DIM = 32
SEQ_LEN = 12
PRED_LEN = 5
EPOCHS = 50
BATCH_SIZE = 50
LR = 0.00005
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')




In [19]:
# -------------------------
# Step 2: Dataset Preparation (Run Second)
# -------------------------
class CSIDataset(Dataset):
    def __init__(self, file_path):
        self.df = pd.read_csv(file_path, header=None)
        self.scaler = StandardScaler()
        self.data = self.scaler.fit_transform(self.df.values)
        
    def __len__(self):
        return len(self.data) - SEQ_LEN - PRED_LEN

    def __getitem__(self, idx):
        x = self.data[idx:idx+SEQ_LEN]
        y = self.data[idx+SEQ_LEN:idx+SEQ_LEN+PRED_LEN]
        return torch.FloatTensor(x), torch.FloatTensor(y)

# Initialize dataset and loader
dataset = CSIDataset(r"C:\Users\Aftab Dayer\Desktop\Thesis\dataset\Uma_256.csv")
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)



In [20]:
# -------------------------
# Step 3: Optimized Model Architecture (Run Third)
# -------------------------
class GraphAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, heads=4):
        super().__init__()
        self.heads = heads
        self.W = nn.Linear(in_dim, heads*out_dim, bias=False)
        self.a = nn.Parameter(torch.randn(2*out_dim, 1))
        
    def forward(self, h):
        # h shape: [batch, seq, features]
        Wh = self.W(h).view(*h.shape[:2], self.heads, -1)
        e = torch.matmul(Wh, Wh.transpose(2,3)) / np.sqrt(Wh.size(-1))
        attention = F.softmax(e, dim=-1)
        h_prime = torch.matmul(attention, Wh)
        return F.elu(h_prime.view(*h.shape[:2], -1))

# -------------------------
# Model Definitions
# -------------------------
class EnhancedGAT(nn.Module):  # Your Original Model
    def __init__(self):
        super().__init__()
        self.gat1 = GraphAttentionLayer(INPUT_DIM, 64)
        self.gat2 = GraphAttentionLayer(256, 128)
        self.tcn = nn.Conv1d(INPUT_DIM, 512, kernel_size=3, padding=1)
        self.fc = nn.Sequential(
            nn.Linear(512 + 512, 1024),
            nn.ReLU(),
            nn.Linear(1024, OUTPUT_DIM)
        )
    
    def forward(self, x):
        # x shape: [batch, seq=12, features=256]
        x_gat = self.gat1(x)  # [50, 12, 256]
        x_gat = self.gat2(x_gat)  # [50, 12, 512]
        
        x_tcn = self.tcn(x.permute(0,2,1))  # [50, 256, 12] → [50, 512, 12]
        x_tcn = x_tcn.permute(0,2,1)  # [50, 12, 512]
        
        combined = torch.cat([x_gat, x_tcn], dim=-1)  # [50, 12, 1024]
        return self.fc(combined[:, -1, :])  # [50, 32]

    # Keep original forward() unchanged

class ComparativeLSTM(nn.Module):  # New Comparison Model
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(INPUT_DIM, 512, num_layers=3, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, OUTPUT_DIM)
        )
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class TemporalCNN(nn.Module):  # Additional Baseline
    def __init__(self):
        super().__init__()
        self.tcn = nn.Sequential(
            nn.Conv1d(SEQ_LEN, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(128*INPUT_DIM, OUTPUT_DIM)
        )
    
    def forward(self, x):
        return self.tcn(x.permute(0,2,1)).squeeze()



In [21]:
def spectral_efficiency(y_true, y_pred, snr_db=30):
    # Convert tensors to numpy arrays
    if isinstance(y_true, torch.Tensor):
        y_true = y_true.cpu().numpy()
    if isinstance(y_pred, torch.Tensor):
        y_pred = y_pred.cpu().numpy()
    
    # Reshape to (batch_size, 16, 2) for real/imaginary components
    if y_true.ndim == 2:
        y_true = y_true.reshape(-1, 16, 2)
    if y_pred.ndim == 2:
        y_pred = y_pred.reshape(-1, 16, 2)
    
    # Create complex representations
    H_true = y_true[..., 0] + 1j*y_true[..., 1]
    H_pred = y_pred[..., 0] + 1j*y_pred[..., 1]
    
    # Calculate correlation (fixed axis handling)
    numerator = np.abs(np.sum(H_pred * np.conj(H_true), axis=1))**2
    denominator = np.sum(np.abs(H_true)**2, axis=1) * np.sum(np.abs(H_pred)**2, axis=1)
    correlation = np.mean(numerator / (denominator + 1e-9))
    
    # Calculate SINR
    noise_var = 10 ** (-snr_db / 10)
    sinr = correlation / (1 - correlation + noise_var)
    return np.log2(1 + sinr)


In [22]:
# -------------------------
# Enhanced Training Loop
# -------------------------
def train_comparative_models():
    models = {
        'Your GAT': EnhancedGAT().to(DEVICE),
        'Comparative LSTM': ComparativeLSTM().to(DEVICE),
        'Temporal CNN': TemporalCNN().to(DEVICE)
    }
    
    results = {}
    
    for model_name, model in models.items():
        print(f"\n=== Training {model_name} ===")
        optimizer = optim.RMSprop(model.parameters(), lr=LR)
        metrics = {'train_loss': [], 'val_rmse': [], 'val_se': []}
        
        for epoch in range(EPOCHS):
            # Training phase
            model.train()
            epoch_loss = 0
            for x, y in loader:
                x, y = x.to(DEVICE), y[:, -1, :OUTPUT_DIM].to(DEVICE)
                optimizer.zero_grad()
                pred = model(x)
                loss = criterion(pred, y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            # Validation phase
            model.eval()
            rmse, se = 0, 0
            with torch.no_grad():
                for x, y in loader:
                    x = x.to(DEVICE)
                    y_true = y[:, -1, :OUTPUT_DIM].cpu()
                    pred = model(x).cpu()
                    rmse += np.sqrt(mean_squared_error(y_true, pred))
                    se += spectral_efficiency(y_true, pred)
            
            metrics['train_loss'].append(epoch_loss/len(loader))
            metrics['val_rmse'].append(rmse/len(loader))
            metrics['val_se'].append(se/len(loader))
            
            print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {metrics['train_loss'][-1]:.4f} | RMSE: {metrics['val_rmse'][-1]:.4f} | SE: {metrics['val_se'][-1]:.4f}")
        
        results[model_name] = metrics
        torch.save(model.state_dict(), f'results/{model_name.replace(" ", "_")}.pth')
    
    return results



In [23]:
# -------------------------
# Enhanced Visualization
# -------------------------
def plot_comparative_results(results):
    plt.figure(figsize=(15, 5))
    
    # RMSE Comparison
    plt.subplot(1, 2, 1)
    for model_name, metrics in results.items():
        plt.plot(metrics['val_rmse'], label=model_name)
    plt.xlabel('Epochs')
    plt.ylabel('RMSE')
    plt.title('Validation RMSE Comparison')
    plt.legend()
    
    # Spectral Efficiency Comparison
    plt.subplot(1, 2, 2)
    for model_name, metrics in results.items():
        plt.plot(metrics['val_se'], label=model_name)
    plt.xlabel('Epochs')
    plt.ylabel('Spectral Efficiency')
    plt.title('Spectral Efficiency Comparison')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig('results/model_comparison.png', dpi=300)
    plt.show()



In [24]:
# -------------------------
# Execution Flow
# -------------------------
if __name__ == "__main__":
    # Initialize dataset
    dataset = CSIDataset(r"C:\Users\Aftab Dayer\Desktop\Thesis\dataset\Uma_256.csv")
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    criterion = nn.MSELoss()
    
    # Train all models
    results = train_comparative_models()
    
    # Save and plot results
    pd.DataFrame({k: v['val_se'] for k,v in results.items()}).to_csv('results/se_comparison.csv')
    pd.DataFrame({k: v['val_rmse'] for k,v in results.items()}).to_csv('results/rmse_comparison.csv')
    plot_comparative_results(results)



=== Training Your GAT ===
Epoch 1/50 | Loss: 0.2698 | RMSE: 0.3129 | SE: 3.4327
Epoch 2/50 | Loss: 0.0612 | RMSE: 0.1905 | SE: 4.7744
Epoch 3/50 | Loss: 0.0282 | RMSE: 0.1457 | SE: 5.5012
Epoch 4/50 | Loss: 0.0185 | RMSE: 0.1241 | SE: 5.9373
Epoch 5/50 | Loss: 0.0141 | RMSE: 0.1103 | SE: 6.2540
Epoch 6/50 | Loss: 0.0116 | RMSE: 0.1024 | SE: 6.4466
Epoch 7/50 | Loss: 0.0099 | RMSE: 0.0946 | SE: 6.6548
Epoch 8/50 | Loss: 0.0087 | RMSE: 0.0896 | SE: 6.7978
Epoch 9/50 | Loss: 0.0077 | RMSE: 0.0846 | SE: 6.9428
Epoch 10/50 | Loss: 0.0070 | RMSE: 0.0809 | SE: 7.0563
Epoch 11/50 | Loss: 0.0064 | RMSE: 0.0766 | SE: 7.1884
Epoch 12/50 | Loss: 0.0059 | RMSE: 0.0759 | SE: 7.2173
Epoch 13/50 | Loss: 0.0055 | RMSE: 0.0739 | SE: 7.2901
Epoch 14/50 | Loss: 0.0051 | RMSE: 0.0691 | SE: 7.4434
Epoch 15/50 | Loss: 0.0048 | RMSE: 0.0745 | SE: 7.2658
Epoch 16/50 | Loss: 0.0046 | RMSE: 0.0673 | SE: 7.5099
Epoch 17/50 | Loss: 0.0043 | RMSE: 0.0651 | SE: 7.5952
Epoch 18/50 | Loss: 0.0041 | RMSE: 0.0630 | SE:

RuntimeError: Given groups=1, weight of size [64, 12, 3], expected input[50, 256, 12] to have 12 channels, but got 256 channels instead

In [ ]:

# -------------------------
# Modified Comparison Code
# -------------------------
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import os

# Configuration (Keep Original)
MOBILITY = 120
COMPRESSION_RATIO = '1/8'
INPUT_DIM = 256
OUTPUT_DIM = 32
SEQ_LEN = 12
PRED_LEN = 5
EPOCHS = 50
BATCH_SIZE = 50
LR = 0.00005
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------------
# Model Definitions
# -------------------------
class EnhancedGAT(nn.Module):  # Your Original Model
    def __init__(self):
        super().__init__()
        self.gat1 = GraphAttentionLayer(INPUT_DIM, 64)
        self.gat2 = GraphAttentionLayer(256, 128)
        self.tcn = nn.Conv1d(INPUT_DIM, 512, kernel_size=3, padding=1)
        self.fc = nn.Sequential(
            nn.Linear(512 + 512, 1024),
            nn.ReLU(),
            nn.Linear(1024, OUTPUT_DIM)
        )
    
    # Keep original forward() unchanged

class ComparativeLSTM(nn.Module):  # New Comparison Model
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(INPUT_DIM, 512, num_layers=3, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, OUTPUT_DIM)
        )
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class TemporalCNN(nn.Module):  # Additional Baseline
    def __init__(self):
        super().__init__()
        self.tcn = nn.Sequential(
            nn.Conv1d(SEQ_LEN, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(128*INPUT_DIM, OUTPUT_DIM)
        )
    
    def forward(self, x):
        return self.tcn(x.permute(0,2,1)).squeeze()


def spectral_efficiency(y_true, y_pred, snr_db=30):
    # Convert tensors to numpy arrays
    if isinstance(y_true, torch.Tensor):
        y_true = y_true.cpu().numpy()
    if isinstance(y_pred, torch.Tensor):
        y_pred = y_pred.cpu().numpy()
    
    # Reshape to (batch_size, 16, 2) for real/imaginary components
    if y_true.ndim == 2:
        y_true = y_true.reshape(-1, 16, 2)
    if y_pred.ndim == 2:
        y_pred = y_pred.reshape(-1, 16, 2)
    
    # Create complex representations
    H_true = y_true[..., 0] + 1j*y_true[..., 1]
    H_pred = y_pred[..., 0] + 1j*y_pred[..., 1]
    
    # Calculate correlation (fixed axis handling)
    numerator = np.abs(np.sum(H_pred * np.conj(H_true), axis=1))**2
    denominator = np.sum(np.abs(H_true)**2, axis=1) * np.sum(np.abs(H_pred)**2, axis=1)
    correlation = np.mean(numerator / (denominator + 1e-9))
    
    # Calculate SINR
    noise_var = 10 ** (-snr_db / 10)
    sinr = correlation / (1 - correlation + noise_var)
    return np.log2(1 + sinr)

# -------------------------
# Enhanced Training Loop
# -------------------------
def train_comparative_models():
    models = {
        'Your GAT': EnhancedGAT().to(DEVICE),
        'Comparative LSTM': ComparativeLSTM().to(DEVICE),
        'Temporal CNN': TemporalCNN().to(DEVICE)
    }
    
    results = {}
    
    for model_name, model in models.items():
        print(f"\n=== Training {model_name} ===")
        optimizer = optim.RMSprop(model.parameters(), lr=LR)
        metrics = {'train_loss': [], 'val_rmse': [], 'val_se': []}
        
        for epoch in range(EPOCHS):
            # Training phase
            model.train()
            epoch_loss = 0
            for x, y in loader:
                x, y = x.to(DEVICE), y[:, -1, :OUTPUT_DIM].to(DEVICE)
                optimizer.zero_grad()
                pred = model(x)
                loss = criterion(pred, y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            # Validation phase
            model.eval()
            rmse, se = 0, 0
            with torch.no_grad():
                for x, y in loader:
                    x = x.to(DEVICE)
                    y_true = y[:, -1, :OUTPUT_DIM].cpu()
                    pred = model(x).cpu()
                    rmse += np.sqrt(mean_squared_error(y_true, pred))
                    se += spectral_efficiency(y_true, pred)
            
            metrics['train_loss'].append(epoch_loss/len(loader))
            metrics['val_rmse'].append(rmse/len(loader))
            metrics['val_se'].append(se/len(loader))
            
            print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {metrics['train_loss'][-1]:.4f} | RMSE: {metrics['val_rmse'][-1]:.4f} | SE: {metrics['val_se'][-1]:.4f}")
        
        results[model_name] = metrics
        torch.save(model.state_dict(), f'results/{model_name.replace(" ", "_")}.pth')
    
    return results

# -------------------------
# Enhanced Visualization
# -------------------------
def plot_comparative_results(results):
    plt.figure(figsize=(15, 5))
    
    # RMSE Comparison
    plt.subplot(1, 2, 1)
    for model_name, metrics in results.items():
        plt.plot(metrics['val_rmse'], label=model_name)
    plt.xlabel('Epochs')
    plt.ylabel('RMSE')
    plt.title('Validation RMSE Comparison')
    plt.legend()
    
    # Spectral Efficiency Comparison
    plt.subplot(1, 2, 2)
    for model_name, metrics in results.items():
        plt.plot(metrics['val_se'], label=model_name)
    plt.xlabel('Epochs')
    plt.ylabel('Spectral Efficiency')
    plt.title('Spectral Efficiency Comparison')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig('results/model_comparison.png', dpi=300)
    plt.show()

# -------------------------
# Execution Flow
# -------------------------
if __name__ == "__main__":
    # Initialize dataset
    dataset = CSIDataset(r"C:\Users\Aftab Dayer\Desktop\Thesis\dataset\Uma_256.csv")
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    criterion = nn.MSELoss()
    
    # Train all models
    results = train_comparative_models()
    
    # Save and plot results
    pd.DataFrame({k: v['val_se'] for k,v in results.items()}).to_csv('results/se_comparison.csv')
    pd.DataFrame({k: v['val_rmse'] for k,v in results.items()}).to_csv('results/rmse_comparison.csv')
    plot_comparative_results(results)
